In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    BooleanType,
    DoubleType,
    StringType,
    StructField,
    StructType,
)

# Initialize Spark session (Colab-compatible)
spark = SparkSession.builder.appName("MetadataControlTable").getOrCreate()

# ---------------------------------------
# STEP 1: Drop table if exists
# ---------------------------------------
spark.sql("DROP TABLE IF EXISTS control_table")

# ---------------------------------------
# STEP 2: Create config DataFrame schema
# ---------------------------------------
schema = StructType(
    [
        StructField("source_name", StringType(), True),
        StructField("entity_name", StringType(), True),
        StructField("source_type", StringType(), True),
        StructField("source_path", StringType(), True),
        StructField("load_type", StringType(), True),
        StructField("watermark_column", StringType(), True),
        StructField("last_processed_value", StringType(), True),
        StructField("mapping_config_path", StringType(), True),
        StructField("transformation_module", StringType(), True),
        StructField("schedule_type", StringType(), True),
        StructField("active_flag", BooleanType(), True),
        StructField("config_version", StringType(), True),
        StructField("data_quality_threshold", DoubleType(), True),
    ]
)

# ---------------------------------------
# STEP 3: Insert sample records
# ---------------------------------------
records = [
    ("carrier_crm", "sales_transactions", "csv", "/data/carrier_crm/", "full", None, None, "/config/carrier_crm_sales.json", None, "daily", True, "v1", 0.9),
    ("carrier_crm", "customers", "csv", "/data/carrier_crm/", "full", None, None, "/config/carrier_crm_customers.json", None, "daily", True, "v1", 0.9),
    ("carrier_crm", "policies", "csv", "/data/carrier_crm/", "full", None, None, "/config/carrier_crm_policies.json", None, "daily", True, "v1", 0.9),
    ("oas_db", "sales_transactions", "db", "jdbc://oas_db", "incremental", "updated_at", None, "/config/oas_sales.json", None, "hourly", True, "v1", 0.9),
    ("oas_db", "customers", "db", "jdbc://oas_db", "incremental", "updated_at", None, "/config/oas_customers.json", None, "hourly", True, "v1", 0.9),
    ("oas_db", "policies", "db", "jdbc://oas_db", "incremental", "updated_at", None, "/config/oas_policies.json", None, "hourly", True, "v1", 0.9),
]

# ---------------------------------------
# STEP 4: Create DataFrame
# ---------------------------------------
control_df = spark.createDataFrame(records, schema=schema)

# ---------------------------------------
# STEP 5: Register table
# ---------------------------------------
control_df.createOrReplaceTempView("control_table")

# OUTPUT: show DataFrame and print final table
print("Control table DataFrame:")
control_df.show(truncate=False)

print("Final table from Spark SQL:")
spark.sql("SELECT * FROM control_table").show(truncate=False)



Control table DataFrame:
+-----------+------------------+-----------+------------------+-----------+----------------+--------------------+----------------------------------+---------------------+-------------+-----------+--------------+----------------------+
|source_name|entity_name       |source_type|source_path       |load_type  |watermark_column|last_processed_value|mapping_config_path               |transformation_module|schedule_type|active_flag|config_version|data_quality_threshold|
+-----------+------------------+-----------+------------------+-----------+----------------+--------------------+----------------------------------+---------------------+-------------+-----------+--------------+----------------------+
|carrier_crm|sales_transactions|csv        |/data/carrier_crm/|full       |NULL            |NULL                |/config/carrier_crm_sales.json    |NULL                 |daily        |true       |v1            |0.9                   |
|carrier_crm|customers         |csv